# Run ARC SFT Strategy Enrichment

Clone the ARC repository from GitLab, install dependencies, and run the SFT strategy enricher on a JSONL dataset.

## 1. Runtime Parameters

Edit these values before running the notebook if needed. For best performance, use a GPU runtime.

In [ ]:
from pathlib import Path

REPO_URL = "https://gitlab.com/beryl.hoe/arc.git"
PROJECT_DIR = Path("/content/arc")
INPUT_PATH = PROJECT_DIR / "data" / "sft_train.jsonl"
OUTPUT_PATH = PROJECT_DIR / "data" / "enriched_sft_train.jsonl"
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 64
MAX_INPUT_TOKENS = 4096
MAX_DOCUMENT_CHARS = 1200
MAX_TOTAL_DOCUMENT_CHARS = 6000
QUESTION_BATCH_SIZE = 2
FORCE_RECLONE = False


## 2. Install System Packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y git git-lfs wget > /dev/null
!git lfs install


## 3. Clone the Repository

In [ ]:
import shutil
import subprocess

if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if PROJECT_DIR.exists():
    print(f"Repository already exists at {PROJECT_DIR}; pulling latest changes.")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

print(f"Project directory: {PROJECT_DIR}")


## 4. Install Python Dependencies

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)


## 5. Preflight Check

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))

preflight = subprocess.run(
    [
        sys.executable,
        "-c",
        "from src.enrich_sft_with_strategies import main; print('python ok')",
    ],
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print("--- STDOUT ---")
print(preflight.stdout)
print("--- STDERR ---")
print(preflight.stderr)
if preflight.returncode != 0:
    raise RuntimeError(f"Preflight failed with exit code {preflight.returncode}")


## 6. Run the Enricher

In [ ]:
cmd = [
    sys.executable,
    "-u",
    "-m",
    "src.enrich_sft_with_strategies",
    "--input-path",
    str(INPUT_PATH),
    "--output-path",
    str(OUTPUT_PATH),
    "--model-name",
    MODEL_NAME,
    "--max-new-tokens",
    str(MAX_NEW_TOKENS),
    "--max-input-tokens",
    str(MAX_INPUT_TOKENS),
    "--max-document-chars",
    str(MAX_DOCUMENT_CHARS),
    "--max-total-document-chars",
    str(MAX_TOTAL_DOCUMENT_CHARS),
    "--question-batch-size",
    str(QUESTION_BATCH_SIZE),
    "--save-executions",
]

print("Running:", " ".join(cmd))
process = subprocess.Popen(
    cmd,
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"Strategy enricher failed with exit code {returncode}")


## 7. Inspect Output

In [ ]:
import json
from collections import Counter

if OUTPUT_PATH.exists():
    records = []
    with OUTPUT_PATH.open('r', encoding='utf-8') as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    counter = Counter(len(record.get('valid_strategies', [])) for record in records)
    print('Distribution of valid strategies:')
    for key in sorted(counter):
        print(f'  {key} strategies: {counter[key]}')

    print('\nSample questions:')
    for record in records[:5]:
        print(f"  {record.get('query_id', '')}: {record.get('valid_strategies', [])}")
else:
    print('scenarios_merged_with_letters.jsonl: missing')
